# VetXRay Dataset — Validation & summary

This notebook verifies the integrity of a downloaded copy of the **VetXRay**
dataset, prints the basic statistics, and renders the summary figures.

It assumes only the released artefacts are available:

1. a folder of DICOM images (`*.dcm`)
2. the annotation spreadsheet (`*.xlsx`)

The checks are grouped as follows:

| Group | What is verified |
|---|---|
| **A. Spreadsheet structure** | required columns, duplicate rows, missing values |
| **B. Controlled vocabularies** | species, projection, quality, finding labels |
| **C. File cross-reference** | every annotated image exists on disk, and every image on disk is annotated |
| **D. DICOM headers** | modality, photometric interpretation, bit depth, image size, pixel spacing |
| **E. Pixel data** | a random subset is fully decoded and compared against the header geometry |
| **F. De-identification** | headers are scanned for identifying tags |

Each check is reported as **PASS**, **WARN**, **FAIL** or **INFO**:

* **FAIL** — the downloaded copy is incomplete or corrupt
* **WARN** — a noteworthy property of the release itself; the data is usable,
  but the caveat should be understood before analysis
* **INFO** — contextual information, no action implied

**Install dependencies** (run once):
```bash
pip install pydicom pandas numpy matplotlib openpyxl
```

> The notebook uses the implementation in **`validate.py`**, which is released
> alongside it and must sit in the same folder. Running `python validate.py
> --dicom-dir ... --xlsx ...` from a terminal performs exactly the same checks.


In [ ]:
import os

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import validate as v   # validate.py must be in the same folder as this notebook

# validate.py selects a head-less backend so it can run from a terminal; switch
# back to the inline one so that the figures are rendered in the notebook
%matplotlib inline

# ── Configuration ─────────────────────────────────────────────────────────────
# Path to the folder containing all released .dcm files
DICOM_DIR = None    # TODO: set this to the path where your DICOM files are located

# Path to the released annotation spreadsheet
XLSX_PATH = None    # TODO: set this to the path of the annotation spreadsheet

# Where the summary figures and the validation report are written
OUTPUT_DIR = "validation_output"

# Number of DICOM headers to read. None reads every file (~2-3 min for the full
# release); set e.g. 500 for a quick partial pass.
MAX_FILES = None

# How many images are fully decoded for the pixel-level check
PIXEL_SAMPLE = 25

assert DICOM_DIR and XLSX_PATH, "Set DICOM_DIR and XLSX_PATH above before continuing."
pd.set_option("display.max_rows", 200)

## 1. Load the released artefacts

The spreadsheet is read as-is; a few normalised helper columns are added
(`species_norm`, `projection_norm`, `quality_norm`, `breed_norm`, `tags_list`,
`n_findings`) so that the checks are insensitive to stray whitespace and
letter case.

In [ ]:
df = v.load_metadata(XLSX_PATH)
disk_files = v.list_dicom_files(DICOM_DIR)

print(f"Spreadsheet rows : {len(df):,}")
print(f"Files on disk    : {len(disk_files):,}")
display(df.head())

## 2. Annotation checks (A, B)

Structure of the spreadsheet, then the controlled vocabularies it draws on.

In [ ]:
log = v.CheckLog()          # collects every check result

v.check_spreadsheet(df, log)
v.check_vocabularies(df, log)

## 3. File cross-reference (C)

Every row of the spreadsheet must resolve to a file on disk. The reverse
direction is reported as a warning: images that carry no annotation row can
still be read, but they are outside the analysed dataset.

In [ ]:
crossref = v.check_file_crossref(df, disk_files, DICOM_DIR, log)

## 4. Image checks (D, E, F)

Every DICOM header is read once. The resulting table drives the header
consistency checks, the de-identification scan and the technical figures
further down. Pixel data is decoded for a random subset, which is the
expensive part of the check and therefore sampled.

In [ ]:
dcm = v.scan_dicom_headers(DICOM_DIR, disk_files, log, max_files=MAX_FILES)

v.check_dicom_consistency(dcm, log)
pixel_stats = v.check_pixel_data(DICOM_DIR, dcm, log, n_sample=PIXEL_SAMPLE)
v.check_anonymisation(dcm, log)

display(dcm.head())

## 5. Basic statistics

In [ ]:
v.print_statistics(df, dcm)

## 6. Summary figures

The figures are written to `OUTPUT_DIR` and displayed inline.

In [ ]:
figures = v.build_figures(df, dcm, OUTPUT_DIR)

for name, fig in figures.items():
    print(name)
    display(fig)
    plt.close(fig)   # displayed explicitly above, so drop it from the figure stack

## 7. Verdict

The full result table is written to `validation_report.csv` next to the
figures. The dataset passes validation when nothing is reported as **FAIL**.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
log.frame.to_csv(os.path.join(OUTPUT_DIR, "validation_report.csv"), index=False)

log.print_summary()
display(log.frame[log.frame["Status"] != "PASS"])